In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone https://github.com/laukamkit/capstone_project_GroupA.git

In [ ]:
%cd capstone_project_GroupA
!git checkout main

In [ ]:
%cd src

In [ ]:
SAVE_PATH = '/content/drive/MyDrive/Colab Notebooks/capstone_project_GroupA/patchtst_tuning'

Run from here if not using Colab.

**Do not use specific_output_dir**

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from datetime import datetime
from ModelFiles.GroupAModels import TransformersModel
from ModelFiles.ModelConfigs import TransformersConfig, HORIZONS, SEEDS
from ModelFiles.ModelEnums import TransformerModelType
from ModelFiles.ModelPlots import *

USE_LOG_TARGET = True
CONTEXT_LENGTHS = [336, 720] # 336 is original. However, the authors used hourly data and we are using half hourly data. so try 720 as well.
EVAL_STEP_SIZE = 48
NUM_EPOCHS = 100
PATIENCE = 10
DEBUG = False
SEEDS = [31415]
PARAMETERS = [(128, 256, 3, 0.2, 0.0001), (128, 256, 3, 0.0, 0.000005), (512, 2048, 5, 0.2, 0.0001), (512, 2048, 5, 0.0, 0.000005)]
for horizon in HORIZONS:
    for context_length in CONTEXT_LENGTHS:
        for seed in SEEDS:
            for d_model, dim_ff, num_encoder_layers, dropout, learning_rate in PARAMETERS:
                patchtst_config = TransformersConfig(
                    task_id=f"patchtst_{datetime.now().strftime('%Y%m%d_%H%M%S')}",
                    model=TransformerModelType.PATCHTST,
                    forecast_horizon=horizon,
                    lookback_window=context_length,
                    used_log_target=USE_LOG_TARGET,
                    target_col= "LOG_TOTALDEMAND" if USE_LOG_TARGET else "TOTALDEMAND",
                    feature_cols=['TEMPERATURE', 'TEMP_SQUARED', 'IS_WEEKEND', 'demand_1_year_ago'],
                    scale=True,
                    date_col='DATETIME',
                    variate='MS',
                    patch_len=16,
                    stride=8,
                    d_model=d_model,
                    num_attention_heads=16,
                    num_encoder_layers=num_encoder_layers,
                    dim_ff=dim_ff,
                    dropout=dropout,
                    dropout_head_fc=dropout,
                    use_gpu=True,
                    time_encoding='timeF',
                    training_epochs=NUM_EPOCHS,
                    batch_size=32,
                    learning_rate=learning_rate,
                    output_attention=False,
                    lradj='TST',
                    patience=PATIENCE,
                    seed=seed,
                    eval_step_size=EVAL_STEP_SIZE,
                    save_test_results=False,
                    debug=DEBUG,
                    save_training_log=True,
                    save_model=False,
                )
                patch_tst_model = TransformersModel(patchtst_config, specific_output_dir=SAVE_PATH)
                patch_tst_model.train_model()
                print("=" * 200)
                print("\n")